# Milestone 2: Build Graph RAG Index
This notebook generates the NetworkX Graph Knowledge Base on the local Kaggle disk (`/kaggle/working/`) using the ContractNLI training split.

In [ ]:
!pip install -q networkx matplotlib

In [ ]:
import json
import os
import networkx as nx
import pickle

# 1. Load the Training Split
train_path = "/kaggle/working/contract-nli/train.json"
if not os.path.exists(train_path):
    train_path = "sample.json" # Fallback for local testing

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

print(f"Loaded {len(train_data.get('documents', []))} documents from {train_path}")

### Structured Graph Engineering
We extract the actual legal clauses as `Clause` nodes and link them to the overarching `Hypothesis` nodes via a directed edge. The edge attribute (`relation`) indicates whether the clause historically `Entailed` or `Contradicted` the hypothesis.
This interconnected structure will allow the GraphRetriever to pull all historical clauses related to a specific legal rule.

In [ ]:
G = nx.DiGraph()
labels_map = train_data.get('labels', {})
chunk_idx = 0

for doc in train_data.get('documents', []):
    doc_id = str(doc['id'])
    text = doc['text']
    doc_spans = doc.get('spans', [])
    annotations = doc.get('annotation_sets', [{}])[0].get('annotations', {})
    
    for hyp_id, ann in annotations.items():
        choice = ann.get('choice')
        span_indices = ann.get('spans', [])
        hypothesis_text = labels_map.get(hyp_id, {}).get('hypothesis', '')
        
        if choice in ['Entailment', 'Contradiction'] and span_indices:
            for idx in span_indices:
                if idx >= len(doc_spans): continue
                span_val = doc_spans[idx]
                
                if isinstance(span_val, list) and len(span_val) == 2:
                    ev_text = text[span_val[0]:span_val[1]]
                elif isinstance(span_val, str):
                    ev_text = span_val
                else:
                    continue
                    
                # Define specific Node IDs
                clause_node_id = f"Clause_{doc_id}_{chunk_idx}"
                hyp_node_id = f"Rule_{hyp_id}"
                
                # Add Clause Node
                G.add_node(clause_node_id, type="Clause", text=ev_text, source_doc=doc_id)
                
                # Add Hypothesis Node (if it doesn't exist yet)
                if not G.has_node(hyp_node_id):
                    G.add_node(hyp_node_id, type="Hypothesis", text=hypothesis_text)
                    
                # Connect the Clause to the Hypothesis 
                G.add_edge(clause_node_id, hyp_node_id, relation=choice)
                
                chunk_idx += 1

print(f"Built Knowledge Graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

### Save Graph Database

In [ ]:
# Save the graph locally so your main retriever pipeline can load it
out_dir = "/kaggle/working/graph_db"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "contractnli_graph.pkl")

with open(out_path, "wb") as f:
    pickle.dump(G, f)
    
print(f"Graph RAG index firmly saved to {out_path}")